In [ ]:
# ============================================================
# Layer-7 Self-Filtered Greedy PatchCore Ablation
# Dataset: Lusitano
#
# NO GAUSSIAN MODEL IS USED.
#
# Self-filtering pipeline:
#   1. Copy dataset from Google Drive to local Colab storage.
#   2. Extract EfficientNet-B5 features[7] candidate patches.
#   3. Build one initial 20,000-patch greedy coreset.
#   4. Use the initial bank to score every training-normal image.
#   5. Remove the highest-scoring suspicious training images.
#   6. Remove candidate patches originating from those images.
#   7. Build a final greedy coreset from the cleaned candidate pool.
#   8. Evaluate the requested filtering/compression ablations.
#
# Requested experiments:
#   Baseline                : 0% filter       + 20,000 bank
#   Filtering only          : 3%, 5%, 10%    + 20,000 bank
#   Compression only        : 0% filter       + 10,000 and 5,000 banks
#   Filtering + compression : 3%, 5%, 10%    + 10,000 and 5,000 banks
#
# Only the final bank is used for thresholding and test inference.
# ============================================================

import gc
import os
import random
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image, ImageFile
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import EfficientNet_B5_Weights, efficientnet_b5
from tqdm.auto import tqdm

from google.colab import drive


# ============================================================
# 1. Mount Google Drive and reproducibility
# ============================================================

drive.mount("/content/drive")

SEED = 42


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()

ImageFile.LOAD_TRUNCATED_IMAGES = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

try:
    torch.set_float32_matmul_precision("medium")
except Exception:
    pass


# ============================================================
# 2. Paths: Drive source -> local Colab dataset
# ============================================================

DRIVE_DATASET_ROOT = Path("/content/drive/MyDrive/<YOUR_DATASET_FOLDER>/Lusitano_Dataset")
LOCAL_DATASET_ROOT = Path("/content/Lusitano_Dataset")

# False: reuse an existing complete local copy.
# True : delete the local copy and copy again from Drive.
RECOPY_LOCAL_DATASET = False

LOCAL_SAVE_DIR = Path(
    "/content/layer7_self_filtered_patchcore_ablation_20k_10k_5k"
)
DRIVE_SAVE_DIR = Path(
    "/content/drive/MyDrive/<YOUR_OUTPUT_FOLDER>/"
    "layer7_self_filtered_patchcore_ablation_20k_10k_5k"
)

LOCAL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_FILENAME = (
    "layer7_self_filtered_patchcore_ablation_20k_10k_5k_results_seed42.csv"
)
TRAIN_RANKING_FILENAME = (
    "layer7_self_filter_train_ranking_20k_10k_5k_seed42.csv"
)
IMAGE_SCORES_FILENAME = (
    "layer7_self_filtered_patchcore_20k_10k_5k_image_scores_seed42.csv"
)

LOCAL_RESULTS_CSV = LOCAL_SAVE_DIR / RESULTS_FILENAME
DRIVE_RESULTS_CSV = DRIVE_SAVE_DIR / RESULTS_FILENAME

LOCAL_TRAIN_RANKING_CSV = (
    LOCAL_SAVE_DIR / TRAIN_RANKING_FILENAME
)
DRIVE_TRAIN_RANKING_CSV = (
    DRIVE_SAVE_DIR / TRAIN_RANKING_FILENAME
)

LOCAL_IMAGE_SCORES_CSV = (
    LOCAL_SAVE_DIR / IMAGE_SCORES_FILENAME
)
DRIVE_IMAGE_SCORES_CSV = (
    DRIVE_SAVE_DIR / IMAGE_SCORES_FILENAME
)


def copy_dataset_to_local(
    drive_root: Path,
    local_root: Path,
    recopy: bool = False,
) -> tuple[Path, float]:
    """Copy the dataset once to local Colab storage."""
    if not drive_root.exists():
        raise FileNotFoundError(
            f"Google Drive dataset not found: {drive_root}"
        )

    expected_train = (
        local_root / "nondefects" / "nondefects"
    )
    expected_test = local_root / "test" / "test"

    if recopy and local_root.exists():
        print("Deleting old local dataset copy:", local_root)
        shutil.rmtree(local_root)

    if expected_train.exists() and expected_test.exists():
        print("Using existing local dataset copy:", local_root)
        return local_root, 0.0

    if local_root.exists():
        print("Removing incomplete local dataset copy:", local_root)
        shutil.rmtree(local_root)

    print("\nCopying dataset to local Colab storage")
    print("Source:", drive_root)
    print("Target:", local_root)

    copy_start = time.perf_counter()
    shutil.copytree(drive_root, local_root)
    copy_time_sec = time.perf_counter() - copy_start

    print(
        f"Dataset copy completed in "
        f"{copy_time_sec:.3f} seconds"
    )

    return local_root, copy_time_sec


DATASET_ROOT, DATASET_COPY_TIME_SEC = copy_dataset_to_local(
    DRIVE_DATASET_ROOT,
    LOCAL_DATASET_ROOT,
    RECOPY_LOCAL_DATASET,
)

TRAIN_GOOD_PATH = (
    DATASET_ROOT / "nondefects" / "nondefects"
)
TEST_ROOT_PATH = DATASET_ROOT / "test" / "test"

if not TRAIN_GOOD_PATH.exists():
    raise FileNotFoundError(
        f"Training folder not found: {TRAIN_GOOD_PATH}"
    )

if not TEST_ROOT_PATH.exists():
    raise FileNotFoundError(
        f"Test folder not found: {TEST_ROOT_PATH}"
    )


# ============================================================
# 3. Fixed model and experiment settings
# ============================================================

METHOD_NAME = "Layer-7 Self-Filtered Greedy PatchCore"
BACKBONE_NAME = "EfficientNet-B5"
FEATURE_LAYER = 7

IMG_SIZE = 448
BATCH_SIZE = 16
NUM_WORKERS = 0

PATCHES_PER_IMAGE = 200
PRE_POOL = 400_000
STREAM_POOL_MARGIN = 50_000

INITIAL_SCORING_BANK_SIZE = 20_000
MAX_FINAL_BANK_SIZE = 20_000

# Final deployment-memory sizes evaluated in the compression ablation.
COMPRESSION_BANK_SIZES = [10_000, 5_000]

CORESET_CHUNK = 40_000
NN_CHUNK = 20_000
THRESH_SAMPLE_IMAGES = 2_000
WARMUP_BATCHES = 3

FILTER_LEVELS = [3.0, 5.0, 10.0]

EXPERIMENTS = [
    {
        "Experiment": "Baseline",
        "Filter_Percent": 0.0,
        "Final_Memory_Size": 20_000,
    },
    *[
        {
            "Experiment": "Filtering only",
            "Filter_Percent": filter_percent,
            "Final_Memory_Size": 20_000,
        }
        for filter_percent in FILTER_LEVELS
    ],
    *[
        {
            "Experiment": "Compression only",
            "Filter_Percent": 0.0,
            "Final_Memory_Size": memory_size,
        }
        for memory_size in COMPRESSION_BANK_SIZES
    ],
    *[
        {
            "Experiment": "Filtering + compression",
            "Filter_Percent": filter_percent,
            "Final_Memory_Size": memory_size,
        }
        for memory_size in COMPRESSION_BANK_SIZES
        for filter_percent in FILTER_LEVELS
    ],
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE != "cuda":
    raise RuntimeError(
        "Enable an NVIDIA GPU in Colab: "
        "Runtime > Change runtime type > GPU."
    )

print("\nDevice:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print("Dataset root used:", DATASET_ROOT)


# ============================================================
# 4. General helpers
# ============================================================

def bytes_to_mb(value: int | float) -> float:
    return float(value) / (1024.0 ** 2)


def tensor_size_mb(tensor: torch.Tensor) -> float:
    return bytes_to_mb(
        tensor.numel() * tensor.element_size()
    )


def model_size_mb(model: torch.nn.Module) -> float:
    total_bytes = 0

    for parameter in model.parameters():
        total_bytes += (
            parameter.numel()
            * parameter.element_size()
        )

    for buffer in model.buffers():
        total_bytes += (
            buffer.numel()
            * buffer.element_size()
        )

    return bytes_to_mb(total_bytes)


def clear_memory() -> None:
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def reset_peak_gpu_memory() -> None:
    clear_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()


def peak_gpu_memory_mb() -> float:
    torch.cuda.synchronize()
    return bytes_to_mb(
        torch.cuda.max_memory_allocated()
    )


IMG_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
}


def list_images(
    folder: Path,
    recursive: bool = True,
) -> list[Path]:
    iterator = (
        folder.rglob("*")
        if recursive
        else folder.glob("*")
    )

    return [
        path
        for path in sorted(iterator)
        if path.is_file()
        and path.suffix.lower() in IMG_EXTENSIONS
    ]


# ============================================================
# 5. Dataset classes and local DataLoaders
# ============================================================

transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


class ImagePathDataset(Dataset):
    def __init__(
        self,
        paths: list[Path],
        image_transform,
    ) -> None:
        self.paths = list(paths)
        self.image_transform = image_transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, index: int):
        path = self.paths[index]

        with Image.open(path) as image:
            image = image.convert("RGB")
            tensor = self.image_transform(image)

        return tensor, int(index), str(path)


class TestImageDataset(Dataset):
    def __init__(
        self,
        items: list[tuple[Path, int]],
        image_transform,
    ) -> None:
        self.items = list(items)
        self.image_transform = image_transform

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, index: int):
        path, label = self.items[index]

        with Image.open(path) as image:
            image = image.convert("RGB")
            tensor = self.image_transform(image)

        return tensor, int(label), str(path)


def make_image_loader(
    paths: list[Path],
    batch_size: int = BATCH_SIZE,
) -> DataLoader:
    return DataLoader(
        ImagePathDataset(paths, transform),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
    )


def make_test_loader(
    items: list[tuple[Path, int]],
    batch_size: int = BATCH_SIZE,
) -> DataLoader:
    return DataLoader(
        TestImageDataset(items, transform),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
    )


train_paths = list_images(TRAIN_GOOD_PATH)


def folder_to_label(folder_name: str):
    normalized_name = (
        folder_name.lower()
        .replace("_", "-")
        .strip()
    )

    if normalized_name == "non-defects":
        return 0

    if normalized_name == "defects":
        return 1

    return None


test_items = []

for folder in sorted(TEST_ROOT_PATH.iterdir()):
    if not folder.is_dir():
        continue

    label = folder_to_label(folder.name)

    if label is None:
        print("Skipping unknown test folder:", folder.name)
        continue

    for image_path in list_images(folder):
        test_items.append((image_path, label))

if not train_paths:
    raise RuntimeError("No training images were found.")

if not test_items:
    raise RuntimeError("No test images were found.")

test_loader = make_test_loader(test_items)

print("\nDataset summary")
print("Training normal images:", len(train_paths))
print("Test images:", len(test_items))
print(
    "Normal test images:",
    sum(1 for _, label in test_items if label == 0),
)
print(
    "Defect test images:",
    sum(1 for _, label in test_items if label == 1),
)


# ============================================================
# 6. EfficientNet-B5 features[7] extractor
# ============================================================

class EfficientNetB5Layer7Extractor(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.model = efficientnet_b5(
            weights=EfficientNet_B5_Weights.DEFAULT
        )
        self.model.eval()

        for parameter in self.model.parameters():
            parameter.requires_grad = False

        self.feature_map = None

        self.hook_handle = (
            self.model.features[FEATURE_LAYER]
            .register_forward_hook(self._hook)
        )

    def _hook(
        self,
        module,
        inputs,
        output,
    ) -> None:
        self.feature_map = output

    @torch.no_grad()
    def forward(
        self,
        images: torch.Tensor,
    ) -> torch.Tensor:
        self.feature_map = None
        _ = self.model(images)

        if self.feature_map is None:
            raise RuntimeError(
                "No feature map was captured from features[7]."
            )

        feature_map = self.feature_map
        batch, channels, height, width = feature_map.shape

        return (
            feature_map
            .reshape(batch, channels, height * width)
            .permute(0, 2, 1)
            .contiguous()
        )

    def remove_hook(self) -> None:
        self.hook_handle.remove()


backbone = (
    EfficientNetB5Layer7Extractor()
    .to(DEVICE)
    .eval()
)

with torch.no_grad():
    dummy = torch.zeros(
        1,
        3,
        IMG_SIZE,
        IMG_SIZE,
        device=DEVICE,
    )

    dummy_features = backbone(dummy)

FEATURE_DIMENSION = int(dummy_features.shape[-1])
PATCH_GRID_COUNT = int(dummy_features.shape[1])

del dummy, dummy_features
clear_memory()

BACKBONE_SIZE_MB = model_size_mb(backbone)

print("\nBackbone configuration")
print("Backbone:", BACKBONE_NAME)
print("Feature layer: features[7]")
print("Feature dimension:", FEATURE_DIMENSION)
print("Patch grid count:", PATCH_GRID_COUNT)
print(f"Backbone size: {BACKBONE_SIZE_MB:.3f} MB")


# ============================================================
# 7. Candidate-pool extraction with source image IDs
# ============================================================

def compact_candidate_pool(
    feature_chunks: list[torch.Tensor],
    image_id_chunks: list[torch.Tensor],
    key_chunks: list[torch.Tensor],
    maximum_size: int,
):
    features = torch.cat(feature_chunks, dim=0)
    image_ids = torch.cat(image_id_chunks, dim=0)
    random_keys = torch.cat(key_chunks, dim=0)

    if features.shape[0] > maximum_size:
        retained_indices = torch.topk(
            random_keys,
            k=maximum_size,
            largest=False,
        ).indices

        features = features[retained_indices].contiguous()
        image_ids = image_ids[retained_indices].contiguous()
        random_keys = random_keys[retained_indices].contiguous()

    return (
        [features],
        [image_ids],
        [random_keys],
        int(features.shape[0]),
    )


@torch.no_grad()
def extract_candidate_pool(
    model: torch.nn.Module,
    paths: list[Path],
):
    """
    Extract sampled Layer-7 patches and retain the source image ID
    for every candidate patch. This allows suspicious images and all
    patches originating from those images to be removed later.
    """
    set_seed()

    loader = make_image_loader(paths)

    cpu_random_generator = torch.Generator(
        device="cpu"
    )
    cpu_random_generator.manual_seed(SEED + 777)

    feature_chunks = []
    image_id_chunks = []
    key_chunks = []

    retained_patch_count = 0
    total_sampled_patches = 0

    print("\nExtracting candidate Layer-7 patches...")
    extraction_start = time.perf_counter()

    global_image_offset = 0

    for images, local_indices, _ in tqdm(
        loader,
        desc="Candidate patch extraction",
    ):
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        features = model(images)
        batch_size, patch_count, _ = features.shape

        sampled_feature_list = []
        sampled_image_id_list = []

        for batch_index in range(batch_size):
            sample_count = min(
                PATCHES_PER_IMAGE,
                patch_count,
            )

            sampled_patch_indices = torch.randperm(
                patch_count,
                device=DEVICE,
            )[:sample_count]

            sampled_features = (
                features[
                    batch_index,
                    sampled_patch_indices,
                ]
                .detach()
                .float()
                .cpu()
            )

            source_image_id = int(
                local_indices[batch_index]
            )

            sampled_feature_list.append(
                sampled_features
            )

            sampled_image_id_list.append(
                torch.full(
                    (sample_count,),
                    source_image_id,
                    dtype=torch.long,
                )
            )

        batch_features = torch.cat(
            sampled_feature_list,
            dim=0,
        )

        batch_image_ids = torch.cat(
            sampled_image_id_list,
            dim=0,
        )

        current_patch_count = int(
            batch_features.shape[0]
        )

        random_keys = torch.rand(
            current_patch_count,
            generator=cpu_random_generator,
            dtype=torch.float32,
        )

        feature_chunks.append(batch_features)
        image_id_chunks.append(batch_image_ids)
        key_chunks.append(random_keys)

        retained_patch_count += current_patch_count
        total_sampled_patches += current_patch_count
        global_image_offset += batch_size

        if (
            retained_patch_count
            > PRE_POOL + STREAM_POOL_MARGIN
        ):
            (
                feature_chunks,
                image_id_chunks,
                key_chunks,
                retained_patch_count,
            ) = compact_candidate_pool(
                feature_chunks,
                image_id_chunks,
                key_chunks,
                PRE_POOL,
            )

        del images, features
        del sampled_feature_list
        del sampled_image_id_list
        del batch_features, batch_image_ids, random_keys

    (
        feature_chunks,
        image_id_chunks,
        key_chunks,
        retained_patch_count,
    ) = compact_candidate_pool(
        feature_chunks,
        image_id_chunks,
        key_chunks,
        PRE_POOL,
    )

    candidate_features = feature_chunks[0].contiguous()
    candidate_image_ids = image_id_chunks[0].contiguous()

    extraction_time_sec = (
        time.perf_counter() - extraction_start
    )

    print("Total sampled patches:", total_sampled_patches)
    print(
        "Retained candidate pool:",
        tuple(candidate_features.shape),
    )
    print(
        f"Candidate extraction time: "
        f"{extraction_time_sec:.3f} sec"
    )

    return (
        candidate_features,
        candidate_image_ids,
        extraction_time_sec,
        total_sampled_patches,
    )


(
    candidate_pool,
    candidate_image_ids,
    candidate_extraction_time_sec,
    total_sampled_patches,
) = extract_candidate_pool(
    backbone,
    train_paths,
)


# ============================================================
# 8. Exact greedy coreset selection
# ============================================================

@torch.no_grad()
def greedy_coreset_gpu(
    features_cpu: torch.Tensor,
    maximum_samples: int,
    chunk_size: int = CORESET_CHUNK,
    use_fp16: bool = True,
    seed: int = SEED,
):
    """
    Sequential farthest-first greedy coreset.

    The returned order is nested. Therefore, the first 10,000 rows
    and the first 5,000 rows of a 20,000-row result form the
    corresponding compressed banks for the same pool and seed.
    """
    random.seed(seed)

    total_features, feature_dimension = (
        features_cpu.shape
    )

    if total_features <= maximum_samples:
        return features_cpu.clone(), 0.0

    selection_start = time.perf_counter()

    features_gpu = (
        features_cpu
        .to(DEVICE, non_blocking=True)
        .contiguous()
    )

    if use_fp16:
        features_gpu = features_gpu.half()

    selected_indices = torch.empty(
        maximum_samples,
        dtype=torch.long,
        device=DEVICE,
    )

    first_index = random.randint(
        0,
        total_features - 1,
    )

    selected_indices[0] = first_index
    center = features_gpu[
        first_index:first_index + 1
    ]

    minimum_distances = torch.empty(
        total_features,
        dtype=torch.float32,
        device=DEVICE,
    )

    for start_index in range(
        0,
        total_features,
        chunk_size,
    ):
        end_index = min(
            start_index + chunk_size,
            total_features,
        )

        feature_chunk = features_gpu[
            start_index:end_index
        ]

        squared_distance = (
            feature_chunk - center
        ).float().pow(2).sum(dim=1)

        minimum_distances[
            start_index:end_index
        ] = squared_distance

    for selected_count in tqdm(
        range(1, maximum_samples),
        desc=f"Greedy coreset {maximum_samples}",
    ):
        farthest_index = torch.argmax(
            minimum_distances
        )

        selected_indices[
            selected_count
        ] = farthest_index

        center = features_gpu[
            farthest_index:farthest_index + 1
        ]

        for start_index in range(
            0,
            total_features,
            chunk_size,
        ):
            end_index = min(
                start_index + chunk_size,
                total_features,
            )

            feature_chunk = features_gpu[
                start_index:end_index
            ]

            squared_distance = (
                feature_chunk - center
            ).float().pow(2).sum(dim=1)

            minimum_distances[
                start_index:end_index
            ] = torch.minimum(
                minimum_distances[
                    start_index:end_index
                ],
                squared_distance,
            )

    selected_indices_cpu = selected_indices.cpu()

    selected_features = (
        features_cpu[selected_indices_cpu]
        .float()
        .contiguous()
    )

    del features_gpu
    del selected_indices
    del selected_indices_cpu
    del minimum_distances
    clear_memory()

    selection_time_sec = (
        time.perf_counter() - selection_start
    )

    print(
        f"Greedy coreset construction time: "
        f"{selection_time_sec:.3f} sec"
    )

    return selected_features, selection_time_sec


# ============================================================
# 9. PatchCore nearest-neighbour scoring
# ============================================================

@torch.no_grad()
def image_anomaly_score(
    patch_features_gpu: torch.Tensor,
    memory_bank_gpu: torch.Tensor,
    chunk_size: int = NN_CHUNK,
) -> float:
    """
    PatchCore image score:
    maximum nearest-neighbour distance over all image patches.
    """
    patches = patch_features_gpu.float()

    patch_squared_norm = patches.pow(2).sum(
        dim=1,
        keepdim=True,
    )

    minimum_squared_distance = torch.full(
        (patches.shape[0],),
        float("inf"),
        device=DEVICE,
        dtype=torch.float32,
    )

    for start_index in range(
        0,
        memory_bank_gpu.shape[0],
        chunk_size,
    ):
        memory_chunk = memory_bank_gpu[
            start_index:start_index + chunk_size
        ].float()

        memory_squared_norm = (
            memory_chunk.pow(2)
            .sum(dim=1)
            .unsqueeze(0)
        )

        squared_distance = (
            patch_squared_norm
            + memory_squared_norm
            - 2.0 * (
                patches @ memory_chunk.t()
            )
        )

        squared_distance = (
            squared_distance.clamp_min(0.0)
        )

        minimum_squared_distance = torch.minimum(
            minimum_squared_distance,
            squared_distance.min(dim=1).values,
        )

    return float(
        minimum_squared_distance
        .sqrt()
        .max()
        .item()
    )


@torch.no_grad()
def score_training_images(
    model: torch.nn.Module,
    paths: list[Path],
    memory_bank_gpu: torch.Tensor,
):
    """
    Score every training-normal image using the temporary initial bank.
    Higher scores are treated as more suspicious.
    """
    loader = make_image_loader(paths)

    scores = np.empty(
        len(paths),
        dtype=np.float32,
    )

    print("\nScoring training-normal images for self-filtering...")
    scoring_start = time.perf_counter()

    for images, image_indices, _ in tqdm(
        loader,
        desc="Training self-filter scores",
    ):
        images_gpu = images.to(
            DEVICE,
            non_blocking=True,
        )

        patch_features = model(images_gpu)

        for batch_index in range(
            patch_features.shape[0]
        ):
            image_id = int(
                image_indices[batch_index]
            )

            scores[image_id] = image_anomaly_score(
                patch_features[batch_index],
                memory_bank_gpu,
            )

        del images_gpu, patch_features

    torch.cuda.synchronize()

    scoring_time_sec = (
        time.perf_counter() - scoring_start
    )

    print(
        f"Training-image scoring time: "
        f"{scoring_time_sec:.3f} sec"
    )

    return scores, scoring_time_sec


# ============================================================
# 10. Build temporary initial bank and rank training images
# ============================================================

print("\n" + "=" * 100)
print("Building temporary initial 20,000-patch scoring bank")
print("=" * 100)

initial_ordered_bank_cpu, initial_build_time_sec = (
    greedy_coreset_gpu(
        candidate_pool,
        INITIAL_SCORING_BANK_SIZE,
        CORESET_CHUNK,
        True,
        SEED,
    )
)

initial_scoring_bank_gpu = (
    initial_ordered_bank_cpu
    .to(DEVICE, non_blocking=True)
    .contiguous()
)

train_self_filter_scores, train_scoring_time_sec = (
    score_training_images(
        backbone,
        train_paths,
        initial_scoring_bank_gpu,
    )
)

del initial_scoring_bank_gpu
clear_memory()

train_ranking_base = pd.DataFrame(
    {
        "train_image_id": np.arange(
            len(train_paths),
            dtype=np.int32,
        ),
        "image_path": [
            str(path)
            for path in train_paths
        ],
        "self_filter_score": (
            train_self_filter_scores
        ),
    }
).sort_values(
    "self_filter_score",
    ascending=False,
).reset_index(drop=True)

train_ranking_base["suspicion_rank"] = (
    np.arange(
        1,
        len(train_ranking_base) + 1,
        dtype=np.int32,
    )
)


# ============================================================
# 11. Create clean pools for every filter level
# ============================================================

def get_removed_image_ids(
    ranking_frame: pd.DataFrame,
    filter_percent: float,
) -> set[int]:
    if filter_percent <= 0:
        return set()

    image_count = len(ranking_frame)

    remove_count = int(
        round(
            image_count
            * filter_percent
            / 100.0
        )
    )

    remove_count = max(
        1,
        min(remove_count, image_count - 1),
    )

    return set(
        ranking_frame
        .iloc[:remove_count]["train_image_id"]
        .astype(int)
        .tolist()
    )


FILTERED_POOLS = {
    0.0: candidate_pool,
}
FILTERED_TRAIN_PATHS = {
    0.0: train_paths,
}
FILTER_METADATA = {}
TRAIN_RANKING_FRAMES = []

for filter_percent in [0.0, *FILTER_LEVELS]:
    removed_image_ids = get_removed_image_ids(
        train_ranking_base,
        filter_percent,
    )

    ranking_frame = train_ranking_base.copy()
    ranking_frame["Filter_Percent"] = filter_percent
    ranking_frame["filtered_out"] = (
        ranking_frame["train_image_id"]
        .isin(removed_image_ids)
    )

    TRAIN_RANKING_FRAMES.append(ranking_frame)

    if filter_percent == 0.0:
        cleaned_candidate_pool = candidate_pool
        clean_train_paths = train_paths
    else:
        removed_tensor = torch.zeros(
            len(train_paths),
            dtype=torch.bool,
        )

        if removed_image_ids:
            removed_tensor[
                list(removed_image_ids)
            ] = True

        keep_patch_mask = ~removed_tensor[
            candidate_image_ids
        ]

        cleaned_candidate_pool = (
            candidate_pool[keep_patch_mask]
            .contiguous()
        )

        clean_train_paths = [
            path
            for image_id, path in enumerate(
                train_paths
            )
            if image_id not in removed_image_ids
        ]

        FILTERED_POOLS[
            filter_percent
        ] = cleaned_candidate_pool

        FILTERED_TRAIN_PATHS[
            filter_percent
        ] = clean_train_paths

    FILTER_METADATA[filter_percent] = {
        "Original_Train_Images": len(train_paths),
        "Removed_Train_Images": len(removed_image_ids),
        "Clean_Train_Images": len(clean_train_paths),
        "Original_Candidate_Patches": int(
            candidate_pool.shape[0]
        ),
        "Clean_Candidate_Patches": int(
            cleaned_candidate_pool.shape[0]
        ),
        "Removed_Candidate_Patches": int(
            candidate_pool.shape[0]
            - cleaned_candidate_pool.shape[0]
        ),
    }

    print(
        f"\nFilter {filter_percent:.1f}% | "
        f"removed images={len(removed_image_ids)} | "
        f"clean images={len(clean_train_paths)} | "
        f"clean patches={cleaned_candidate_pool.shape[0]}"
    )


train_ranking_all = pd.concat(
    TRAIN_RANKING_FRAMES,
    ignore_index=True,
)

train_ranking_all.to_csv(
    LOCAL_TRAIN_RANKING_CSV,
    index=False,
)

train_ranking_all.to_csv(
    DRIVE_TRAIN_RANKING_CSV,
    index=False,
)


# ============================================================
# 12. Build one ordered 20,000 bank per filter level; slice to 10,000/5,000
# ============================================================

ORDERED_FINAL_BANKS = {
    0.0: initial_ordered_bank_cpu,
}

FINAL_CORESET_BUILD_TIMES = {
    0.0: initial_build_time_sec,
}

for filter_percent in FILTER_LEVELS:
    print("\n" + "=" * 100)
    print(
        f"Building final clean coreset | "
        f"filter={filter_percent}% | "
        f"maximum bank={MAX_FINAL_BANK_SIZE}"
    )
    print("=" * 100)

    ordered_bank, build_time_sec = (
        greedy_coreset_gpu(
            FILTERED_POOLS[filter_percent],
            MAX_FINAL_BANK_SIZE,
            CORESET_CHUNK,
            True,
            SEED,
        )
    )

    ORDERED_FINAL_BANKS[
        filter_percent
    ] = ordered_bank

    FINAL_CORESET_BUILD_TIMES[
        filter_percent
    ] = build_time_sec


# ============================================================
# 13. Threshold and test evaluation
# ============================================================

@torch.no_grad()
def compute_threshold(
    model: torch.nn.Module,
    clean_train_paths: list[Path],
    memory_bank_gpu: torch.Tensor,
):
    rng = np.random.default_rng(SEED + 100)

    subset_count = min(
        THRESH_SAMPLE_IMAGES,
        len(clean_train_paths),
    )

    selected_indices = rng.permutation(
        len(clean_train_paths)
    )[:subset_count]

    threshold_paths = [
        clean_train_paths[index]
        for index in selected_indices
    ]

    loader = make_image_loader(threshold_paths)
    threshold_scores = []

    threshold_start = time.perf_counter()

    for images, _, _ in tqdm(
        loader,
        desc="Threshold scoring",
    ):
        images_gpu = images.to(
            DEVICE,
            non_blocking=True,
        )

        patch_features = model(images_gpu)

        for batch_index in range(
            patch_features.shape[0]
        ):
            threshold_scores.append(
                image_anomaly_score(
                    patch_features[batch_index],
                    memory_bank_gpu,
                )
            )

        del images_gpu, patch_features

    torch.cuda.synchronize()

    threshold_time_sec = (
        time.perf_counter() - threshold_start
    )

    threshold_scores = np.asarray(
        threshold_scores,
        dtype=np.float32,
    )

    threshold = float(
        threshold_scores.mean()
        + 3.0
        * threshold_scores.std(ddof=1)
    )

    return (
        threshold,
        threshold_scores,
        threshold_time_sec,
    )


@torch.no_grad()
def warm_up(
    model: torch.nn.Module,
    memory_bank_gpu: torch.Tensor,
) -> None:
    first_batch = next(iter(test_loader))

    images_gpu = first_batch[0][:1].to(
        DEVICE,
        non_blocking=True,
    )

    for _ in range(WARMUP_BATCHES):
        patch_features = model(images_gpu)

        _ = image_anomaly_score(
            patch_features[0],
            memory_bank_gpu,
        )

    torch.cuda.synchronize()

    del images_gpu, patch_features
    clear_memory()


@torch.no_grad()
def evaluate_test_set(
    model: torch.nn.Module,
    memory_bank_gpu: torch.Tensor,
    threshold: float,
):
    warm_up(model, memory_bank_gpu)
    reset_peak_gpu_memory()

    true_labels = []
    anomaly_scores = []
    image_paths = []

    compute_total_time_sec = 0.0

    print("\nEvaluating test set...")
    end_to_end_start = time.perf_counter()

    for images, labels, paths in tqdm(
        test_loader,
        desc="Testing",
    ):
        # Data loading and preprocessing have completed before this point.
        torch.cuda.synchronize()
        compute_start = time.perf_counter()

        images_gpu = images.to(
            DEVICE,
            non_blocking=True,
        )

        patch_features = model(images_gpu)

        batch_scores = [
            image_anomaly_score(
                patch_features[batch_index],
                memory_bank_gpu,
            )
            for batch_index in range(
                patch_features.shape[0]
            )
        ]

        torch.cuda.synchronize()

        compute_total_time_sec += (
            time.perf_counter() - compute_start
        )

        anomaly_scores.extend(batch_scores)
        true_labels.extend(
            int(label)
            for label in labels
        )
        image_paths.extend(
            str(path)
            for path in paths
        )

        del images_gpu, patch_features, batch_scores

    torch.cuda.synchronize()

    end_to_end_total_time_sec = (
        time.perf_counter() - end_to_end_start
    )

    true_labels_array = np.asarray(
        true_labels,
        dtype=np.int32,
    )

    anomaly_scores_array = np.asarray(
        anomaly_scores,
        dtype=np.float32,
    )

    predicted_labels = (
        anomaly_scores_array > threshold
    ).astype(np.int32)

    auc_roc = roc_auc_score(
        true_labels_array,
        anomaly_scores_array,
    )

    average_precision = average_precision_score(
        true_labels_array,
        anomaly_scores_array,
    )

    f1 = f1_score(
        true_labels_array,
        predicted_labels,
        zero_division=0,
    )

    tn, fp, fn, tp = confusion_matrix(
        true_labels_array,
        predicted_labels,
        labels=[0, 1],
    ).ravel()

    image_count = max(
        len(true_labels_array),
        1,
    )

    compute_time_per_image_sec = (
        compute_total_time_sec / image_count
    )

    end_to_end_time_per_image_sec = (
        end_to_end_total_time_sec / image_count
    )

    throughput_fps = (
        1.0 / end_to_end_time_per_image_sec
        if end_to_end_time_per_image_sec > 0
        else 0.0
    )

    image_score_frame = pd.DataFrame(
        {
            "image_path": image_paths,
            "gt_label": true_labels_array,
            "anomaly_score": anomaly_scores_array,
            "pred_label": predicted_labels,
        }
    )

    metrics = {
        "AUC_ROC": float(auc_roc),
        "mAP_AP": float(average_precision),
        "F1_Score": float(f1),
        "Threshold": float(threshold),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Compute_Inference_Total_sec": float(
            compute_total_time_sec
        ),
        "Compute_Inference_Time_Per_Image_sec": float(
            compute_time_per_image_sec
        ),
        "End_To_End_Total_sec": float(
            end_to_end_total_time_sec
        ),
        "End_To_End_Time_Per_Image_sec": float(
            end_to_end_time_per_image_sec
        ),
        "Throughput_FPS": float(throughput_fps),
        "Peak_GPU_Memory_MB": float(
            peak_gpu_memory_mb()
        ),
    }

    return metrics, image_score_frame


# ============================================================
# 14. Run requested ablation matrix
# ============================================================

all_result_rows = []
all_image_score_frames = []

experiment_start = time.perf_counter()

for experiment_id, experiment in enumerate(
    EXPERIMENTS,
    start=1,
):
    experiment_name = experiment["Experiment"]

    filter_percent = float(
        experiment["Filter_Percent"]
    )

    final_memory_size = int(
        experiment["Final_Memory_Size"]
    )

    print("\n" + "#" * 100)
    print(
        f"Experiment {experiment_id}/{len(EXPERIMENTS)} | "
        f"{experiment_name} | "
        f"filter={filter_percent}% | "
        f"memory={final_memory_size}"
    )
    print("#" * 100)

    ordered_bank = ORDERED_FINAL_BANKS[
        filter_percent
    ]

    if ordered_bank.shape[0] < final_memory_size:
        raise RuntimeError(
            f"Requested {final_memory_size} memory patches, "
            f"but only {ordered_bank.shape[0]} are available."
        )

    final_memory_bank_cpu = (
        ordered_bank[:final_memory_size]
        .clone()
        .float()
        .contiguous()
    )

    final_memory_bank_gpu = (
        final_memory_bank_cpu
        .to(DEVICE, non_blocking=True)
        .contiguous()
    )

    memory_bank_size_mb = tensor_size_mb(
        final_memory_bank_cpu
    )

    total_footprint_mb = (
        BACKBONE_SIZE_MB
        + memory_bank_size_mb
    )

    (
        threshold,
        threshold_scores,
        threshold_time_sec,
    ) = compute_threshold(
        backbone,
        FILTERED_TRAIN_PATHS[
            filter_percent
        ],
        final_memory_bank_gpu,
    )

    metrics, image_scores = evaluate_test_set(
        backbone,
        final_memory_bank_gpu,
        threshold,
    )

    filter_metadata = FILTER_METADATA[
        filter_percent
    ]

    result_row = {
        "Experiment_ID": experiment_id,
        "Experiment": experiment_name,
        "Dataset": "Lusitano",
        "Method": METHOD_NAME,
        "Backbone": BACKBONE_NAME,
        "Feature_Layer": "features[7]",
        "Seed": SEED,
        "IMG_SIZE": IMG_SIZE,
        "Batch_Size": BATCH_SIZE,
        "Feature_Dimension": FEATURE_DIMENSION,
        "Patch_Grid_Count": PATCH_GRID_COUNT,
        "Patches_Per_Image": PATCHES_PER_IMAGE,
        "Pre_Pool_Limit": PRE_POOL,
        "Filter_Percent": filter_percent,
        "Final_Memory_Size": final_memory_size,
        "Original_Train_Images": (
            filter_metadata[
                "Original_Train_Images"
            ]
        ),
        "Removed_Train_Images": (
            filter_metadata[
                "Removed_Train_Images"
            ]
        ),
        "Clean_Train_Images": (
            filter_metadata[
                "Clean_Train_Images"
            ]
        ),
        "Original_Candidate_Patches": (
            filter_metadata[
                "Original_Candidate_Patches"
            ]
        ),
        "Removed_Candidate_Patches": (
            filter_metadata[
                "Removed_Candidate_Patches"
            ]
        ),
        "Clean_Candidate_Patches": (
            filter_metadata[
                "Clean_Candidate_Patches"
            ]
        ),
        "AUC_ROC": metrics["AUC_ROC"],
        "mAP_AP": metrics["mAP_AP"],
        "F1_Score": metrics["F1_Score"],
        "Threshold": metrics["Threshold"],
        "TN": metrics["TN"],
        "FP": metrics["FP"],
        "FN": metrics["FN"],
        "TP": metrics["TP"],
        "Compute_Inference_Time_Per_Image_sec": (
            metrics[
                "Compute_Inference_Time_Per_Image_sec"
            ]
        ),
        "End_To_End_Time_Per_Image_sec": (
            metrics[
                "End_To_End_Time_Per_Image_sec"
            ]
        ),
        "Throughput_FPS": metrics[
            "Throughput_FPS"
        ],
        "Memory_Bank_Size_MB": (
            memory_bank_size_mb
        ),
        "Backbone_Model_Size_MB": (
            BACKBONE_SIZE_MB
        ),
        "Estimated_Total_Footprint_MB": (
            total_footprint_mb
        ),
        "Peak_GPU_Memory_MB": metrics[
            "Peak_GPU_Memory_MB"
        ],
        "Threshold_Time_sec": (
            threshold_time_sec
        ),
        "Initial_Scoring_Bank_Build_Time_sec": (
            initial_build_time_sec
        ),
        "Train_Self_Filter_Scoring_Time_sec": (
            train_scoring_time_sec
        ),
        "Final_Coreset_Build_Time_sec": (
            FINAL_CORESET_BUILD_TIMES[
                filter_percent
            ]
        ),
        "Candidate_Extraction_Time_sec": (
            candidate_extraction_time_sec
        ),
        "Dataset_Copy_Time_sec": (
            DATASET_COPY_TIME_SEC
        ),
    }

    all_result_rows.append(result_row)

    image_scores["Experiment_ID"] = experiment_id
    image_scores["Experiment"] = experiment_name
    image_scores["Filter_Percent"] = filter_percent
    image_scores["Final_Memory_Size"] = (
        final_memory_size
    )

    all_image_score_frames.append(
        image_scores
    )

    current_results = pd.DataFrame(
        all_result_rows
    )

    current_image_scores = pd.concat(
        all_image_score_frames,
        ignore_index=True,
    )

    # Save after each experiment to avoid losing completed results.
    current_results.to_csv(
        LOCAL_RESULTS_CSV,
        index=False,
    )

    current_results.to_csv(
        DRIVE_RESULTS_CSV,
        index=False,
    )

    current_image_scores.to_csv(
        LOCAL_IMAGE_SCORES_CSV,
        index=False,
    )

    current_image_scores.to_csv(
        DRIVE_IMAGE_SCORES_CSV,
        index=False,
    )

    print("\nResult")
    print(f"AUC-ROC: {metrics['AUC_ROC']:.6f}")
    print(f"AP      : {metrics['mAP_AP']:.6f}")
    print(f"F1      : {metrics['F1_Score']:.6f}")
    print(
        "Compute latency:",
        f"{metrics['Compute_Inference_Time_Per_Image_sec'] * 1000:.3f}",
        "ms/image",
    )
    print(
        "End-to-end latency:",
        f"{metrics['End_To_End_Time_Per_Image_sec'] * 1000:.3f}",
        "ms/image",
    )
    print(
        f"Memory bank: "
        f"{memory_bank_size_mb:.3f} MB"
    )
    print(
        f"Total footprint: "
        f"{total_footprint_mb:.3f} MB"
    )

    del final_memory_bank_cpu
    del final_memory_bank_gpu
    del threshold_scores
    clear_memory()


# ============================================================
# 15. Final output tables
# ============================================================

TOTAL_EXPERIMENT_TIME_SEC = (
    time.perf_counter() - experiment_start
)

df_results = pd.DataFrame(
    all_result_rows
)

df_image_scores = pd.concat(
    all_image_score_frames,
    ignore_index=True,
)

df_results[
    "Compute_Latency_ms_per_image"
] = (
    df_results[
        "Compute_Inference_Time_Per_Image_sec"
    ]
    * 1000.0
)

df_results[
    "End_To_End_Latency_ms_per_image"
] = (
    df_results[
        "End_To_End_Time_Per_Image_sec"
    ]
    * 1000.0
)

df_results[
    "Total_Experiment_Time_sec"
] = TOTAL_EXPERIMENT_TIME_SEC

df_results.to_csv(
    LOCAL_RESULTS_CSV,
    index=False,
)

df_results.to_csv(
    DRIVE_RESULTS_CSV,
    index=False,
)

df_image_scores.to_csv(
    LOCAL_IMAGE_SCORES_CSV,
    index=False,
)

df_image_scores.to_csv(
    DRIVE_IMAGE_SCORES_CSV,
    index=False,
)

summary_columns = [
    "Experiment",
    "Filter_Percent",
    "Final_Memory_Size",
    "Removed_Train_Images",
    "Removed_Candidate_Patches",
    "AUC_ROC",
    "mAP_AP",
    "F1_Score",
    "Compute_Latency_ms_per_image",
    "End_To_End_Latency_ms_per_image",
    "Throughput_FPS",
    "Estimated_Total_Footprint_MB",
    "Peak_GPU_Memory_MB",
]

print("\n" + "=" * 110)
print("FINAL SELF-FILTERING ABLATION SUMMARY")
print("=" * 110)

display(
    df_results[summary_columns]
    .sort_values(
        [
            "Final_Memory_Size",
            "Filter_Percent",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print("\nLocal results:")
print(LOCAL_RESULTS_CSV)

print("\nGoogle Drive results:")
print(DRIVE_RESULTS_CSV)

print("\nTraining self-filter ranking:")
print(DRIVE_TRAIN_RANKING_CSV)

print("\nImage-level test scores:")
print(DRIVE_IMAGE_SCORES_CSV)

print(
    f"\nTotal experiment time: "
    f"{TOTAL_EXPERIMENT_TIME_SEC:.3f} seconds"
)